## WorldView Multi-View Stereo Processing: SpaceNet Atlanta Example

This notebook processes **three same-pass WorldView-2 scenes** over Hartsfield–Jackson Atlanta International Airport (2009-12-22, SpaceNet AOI 6) two ways:

1. **A joint 3-scene multi-view stereo (MVS) run** — one `parallel_stereo` call with all three images, triangulating every pixel from up to three rays.
2. **The three pairwise stereo runs merged with `dem_mosaic`** — the workflow the [ASP documentation recommends](https://stereopipeline.readthedocs.io/en/latest/next_steps.html#multi-view-stereo) over MVS.

The final sections compare the two DEMs directly: coverage, DEM-to-DEM difference, and residuals against ICESat-2 ATL06-SR altimetry.

### Scenes

All five downloaded scenes are from the same 2009-12-22 pass (the SpaceNet "off-nadir" collect names give the nominal look angle). The three-scene subset used for stereo was chosen with the companion [scene-selection notebook](https://asp-plot.readthedocs.io/en/latest/examples/notebooks/worldview_spacenet_atlanta_stereo_scene_selection.html); the two extra scenes strengthen the bundle adjustment network:

| catid | tile | SpaceNet collect | off-nadir (°) | GSD (m) | role |
|---|---|---|---|---|---|
| `1030010002B7D800` | P002 | nadir13 | 14.0 | 0.49 | **MVS reference** + stereo |
| `1030010003CAF100` | P002 | nadir10 | 10.6 | 0.48 | stereo |
| `1030010002649200` | P001 | nadir16 | 18.1 | 0.51 | stereo |
| `10300100023BC100` | P001 | nadir8 | ~8 | 0.48 | bundle adjustment only |
| `1030010003127500` | P001 | nadir21 | ~21 | 0.52 | bundle adjustment only |

Pairwise geometry of the stereo trio (from `StereopairMetadataParser`):

- **nadir13–nadir10**: 21.8° convergence, B:H 0.38, asymmetry 2.6° — the strong pair used by the retired two-scene Atlanta example
- **nadir10–nadir16**: 27.1° convergence, B:H 0.48, asymmetry 5.0° — the strongest pair (the two scenes look at the ground from opposite sides of nadir)
- **nadir13–nadir16**: 5.3° convergence, B:H 0.09, asymmetry 15.9° — a weak pair on its own

That mix is deliberate: the joint MVS triangulation can exploit the weak pair's rays alongside the strong ones, while in the pairwise workflow the weak pair produces a noisy standalone DEM that `dem_mosaic` blends in. A broader benchmarking of scene combinations and processing flows (scored against ICESat-2 after `pc_align`) is deferred to [issue #169](https://github.com/uw-cryo/asp_plot/issues/169).

---

### Processing Overview

- **Data Retrieval** — Download the five WorldView-2 PAN tiles from AWS S3
- **Stereo Geometry Analysis** — N-scene overview + per-pair geometry from the camera XMLs
- **CCD Artifact Correction** — `wv_correct` on every scene
- **Bundle Adjustment** — one 5-scene network, shared by every stereo run below
- **Multi-View Stereo** — 3-scene `parallel_stereo` (nadir13 reference) + `point2dem`
- **Pairwise Stereo + `dem_mosaic`** — the three pairs, same bundle adjustment and crops, merged into a second DEM
- **Comparison** — coverage, DEM difference, and ICESat-2 residuals, MVS vs. pairwise mosaic
- **Report** — the standard `asp_plot` PDF report for the multi-view run

---

## Data Retrieval

The SpaceNet Atlanta raw L1B images are public at `s3://spacenet-dataset/AOIs/AOI_6_Atlanta/metadata/<CATID>/<workorder>_<PXXX>_PAN/` (~2 GB per PAN tile). Each CATID is delivered as multiple tiles (`P001`, `P002`, `P003`) and the tiling differs between collects — the tile listed in the table above is the one whose ground coverage contains the airport ROI for that CATID. Download one `<catid> <tile>` pair at a time:

```bash
$ mkdir -p atlanta_mvs && cd atlanta_mvs

# Repeat for each <catid> <tile> row of the scene table:
#   1030010002B7D800 P002, 1030010003CAF100 P002, 1030010002649200 P001,
#   10300100023BC100 P001, 1030010003127500 P001
$ aws s3 --no-sign-request sync \
    s3://spacenet-dataset/AOIs/AOI_6_Atlanta/metadata/<catid>/ \
    images/aws/<catid>/ \
    --exclude "*" --include "*<tile>_PAN/*"

# Copy the TIF and XML to the working directory with short names
$ cp images/aws/<catid>/*<tile>_PAN/*P1BS*<tile>.TIF <catid>_<tile>.tif
$ cp images/aws/<catid>/*<tile>_PAN/*P1BS*<tile>.XML <catid>_<tile>.xml
```

### Stereo Geometry Analysis

Before processing, analyze the acquisition geometry of the stereo trio directly from the camera XMLs. With more than two scenes, `asp_plot` produces an N-scene overview figure plus one figure per pair, each titled with that pair's convergence, B:H, bisector-elevation, and asymmetry angles. Passing the three XMLs as explicit `inputs` scopes the analysis to the stereo scenes (the directory also holds the two bundle-adjustment-only scenes):

In [ ]:
# Set the base directory for your processing
directory = "~/Desktop/asp-plot-examples/atlanta_mvs/"

from asp_plot.stereo_geometry import StereoGeometryPlotter

map_crs = "EPSG:32616"  # UTM 16N for Atlanta

sgp = StereoGeometryPlotter(
    inputs=[
        f"{directory}1030010002B7D800_P002.xml",
        f"{directory}1030010003CAF100_P002.xml",
        f"{directory}1030010002649200_P001.xml",
    ]
)

_ = sgp.stereo_geom_plot()

The skyplot makes the geometry mix obvious: nadir13 and nadir16 sit close together north of zenith (hence their weak 5.3° convergence), while nadir10 looks from the south side, giving both of its pairs strong convergence.

### Satellite Position and Orientation Data

The XML camera metadata also contains ephemeris (position/velocity) and attitude (orientation quaternion) data reported by the satellite during acquisition, plus their covariances. Visualizing this data can help assess the quality of the raw metadata before ASP processing — with N scenes the plot draws one column per scene:

In [ ]:
sgp.satellite_position_orientation_plot()

---

## CCD Artifact Correction

WorldView-2 imagery has subpixel CCD boundary misalignments. The SpaceNet Atlanta data was processed in 2019 (before the May 2022 improvement), so `wv_correct` is required for every scene:

```bash
$ cd atlanta_mvs

$ for scene in 1030010002B7D800_P002 1030010003CAF100_P002 1030010002649200_P001 \
               10300100023BC100_P001 1030010003127500_P001; do
    wv_correct ${scene}.tif ${scene}.xml ${scene}_corr.tif
  done
```

The `*_corr.tif` corrected images are used for all subsequent processing steps.

---

## Bundle Adjustment

Bundle adjust all five scenes together into one self-consistent camera network. Every stereo run below — the multi-view run and each pairwise run — uses this same `ba/run` prefix, so the cameras are **identical across the two workflows** and any DEM differences come from the stereo strategy, not the cameras. The two extra scenes (nadir8, nadir21) add rays to the network and keep it ready for the scene-combination experiments deferred to [#169](https://github.com/uw-cryo/asp_plot/issues/169):

```bash
$ cd atlanta_mvs

$ bundle_adjust \
    --threads 8 \
    --ip-per-image 10000 \
    --tri-weight 0.1 \
    --tri-robust-threshold 0.1 \
    --camera-weight 0 \
    1030010002B7D800_P002_corr.tif 10300100023BC100_P001_corr.tif \
    1030010003CAF100_P002_corr.tif 1030010002649200_P001_corr.tif \
    1030010003127500_P001_corr.tif \
    1030010002B7D800_P002.xml 10300100023BC100_P001.xml \
    1030010003CAF100_P002.xml 1030010002649200_P001.xml \
    1030010003127500_P001.xml \
    -o ba/run
```

### Bundle Adjustment Results

Visualize the residuals to verify the optimization reduced reprojection errors (median residual drops from 1.17 px to 0.21 px):

In [ ]:
import contextily as ctx
from asp_plot.bundle_adjust import ReadBundleAdjustFiles, PlotBundleAdjustFiles

bundle_adjust_directory = "ba/"
stereo_directory = "stereo_mvs3/"

ctx_kwargs = {
    "crs": map_crs,
    "source": ctx.providers.Esri.WorldImagery,
    "attribution_size": 0,
    "alpha": 0.5,
}

ba_files = ReadBundleAdjustFiles(directory, bundle_adjust_directory)
resid_initial_gdf, resid_final_gdf = ba_files.get_initial_final_residuals_gdfs(residuals_in_meters=True)

plotter = PlotBundleAdjustFiles(
    [resid_initial_gdf, resid_final_gdf],
    lognorm=True,
    title="Bundle Adjust Initial and Final Residuals",
)
plotter.plot_n_gdfs(
    column_name="mean_residual",
    cbar_label="Mean residual (px)",
    map_crs=map_crs,
    **ctx_kwargs,
)

---

## Multi-View Stereo: Three Scenes

ASP supports true multi-view stereo: pass N images and N cameras, and the **first image is the reference** — disparities are computed from it to every other image, followed by a joint triangulation into a single point cloud. Here nadir13 is the reference. (The [ASP documentation discourages this mode](https://stereopipeline.readthedocs.io/en/latest/next_steps.html#multi-view-stereo) in favor of pairwise runs merged with `dem_mosaic` — which is exactly the alternative we build and compare below. A same-pass collect is the best case for MVS: identical illumination and rigid relative geometry.)

The full scenes are large, so we restrict correlation to an airport ROI with `--left-image-crop-win` (pixel coordinates on the reference image). The window was derived by inverse-projecting the four corners of the airport's UTM box `735685 3722295 741350 3727695` (EPSG:32616) through the reference tile's RPC model — the original `.tif` carries RPC metadata; the `wv_correct` output does not — with a 500 px buffer:

```python
from osgeo import gdal, osr

gdal.UseExceptions()

utm = osr.SpatialReference(); utm.ImportFromEPSG(32616)
geo = osr.SpatialReference(); geo.ImportFromEPSG(4326)
for srs in (utm, geo):
    srs.SetAxisMappingStrategy(osr.OAMS_TRADITIONAL_GIS_ORDER)
ct = osr.CoordinateTransformation(utm, geo)

ds = gdal.Open("1030010002B7D800_P002.tif")  # has RPC metadata
tx = gdal.Transformer(ds, None, ["METHOD=RPC", "RPC_HEIGHT=300"])

xs, ys = [], []
for x, y in [(735685, 3722295), (741350, 3722295),
             (741350, 3727695), (735685, 3727695)]:
    lon, lat, _ = ct.TransformPoint(x, y)
    _, (px, py, _) = tx.TransformPoint(1, lon, lat, 300.0)
    xs.append(px); ys.append(py)

ulx, uly = int(min(xs)) - 500, int(min(ys)) - 500
lrx, lry = int(max(xs)) + 500, int(max(ys)) + 500
print(f"--left-image-crop-win {ulx} {uly} {lrx - ulx} {lry - uly}")
# => --left-image-crop-win 5879 13107 12981 11894
```

Run the multi-view stereo (non-mapprojected images, `--alignment-method affineepipolar`) and grid the joint point cloud at ~4× the input GSD:

```bash
$ cd atlanta_mvs

$ parallel_stereo \
    --stereo-algorithm asp_mgm --subpixel-mode 9 \
    --alignment-method affineepipolar \
    --left-image-crop-win 5879 13107 12981 11894 \
    --bundle-adjust-prefix ba/run \
    1030010002B7D800_P002_corr.tif 1030010003CAF100_P002_corr.tif 1030010002649200_P001_corr.tif \
    1030010002B7D800_P002.xml 1030010003CAF100_P002.xml 1030010002649200_P001.xml \
    stereo_mvs3/run

$ point2dem --tr 1.9 --t_srs EPSG:32616 --errorimage stereo_mvs3/run-PC.tif
```

### Multi-View Results

ASP keeps the per-pair intermediate products of a multi-view run (aligned scenes, match points, disparity maps) in `run-pair*/` subdirectories, with only the joint products (point cloud, DEM, triangulation error) at the top level. `asp_plot` detects this layout and renders the scenes, match points, and disparity **one figure per pair** — the reference image is the same in every pair:

In [ ]:
from asp_plot.scenes import ScenePlotter

scene_plotter = ScenePlotter(directory, stereo_directory, title="Input Scenes")
scene_plotter.plot_scenes()

In [ ]:
from asp_plot.stereo import StereoPlotter

stereo_plotter = StereoPlotter(directory, stereo_directory)
stereo_plotter.title = "Stereo Match Points"
stereo_plotter.plot_match_points()

In [ ]:
stereo_plotter.title = "Disparity (pixels)"
stereo_plotter.plot_disparity(unit="pixels", quiver=True)

The joint products of the multi-view run — the DEM with triangulation error, and detailed hillshades:

In [ ]:
stereo_plotter.title = "Multi-View Stereo DEM Results (nadir13 reference, 3 scenes)"
stereo_plotter.plot_dem_results()

In [ ]:
stereo_plotter.title = "Hillshade with Details"
stereo_plotter.plot_detailed_hillshade(subset_km=1.0)

---

## Pairwise Stereo + `dem_mosaic`

Now the workflow the ASP documentation recommends instead of MVS: run each of the three pairs through `parallel_stereo` independently — **same bundle adjustment prefix, same settings** — grid each point cloud with `point2dem`, and blend the three DEMs with `dem_mosaic`.

One detail needs care. `--left-image-crop-win` is given in *left-image pixel coordinates*. For the two pairs whose left image is the reference (nadir13–nadir10, nadir13–nadir16) we reuse the reference crop window above. The nadir10–nadir16 pair does not include the reference, so we derive an equivalent window on nadir10 from the bundle-adjustment clean match points: take the nadir13↔nadir10 matches, keep those whose nadir13 pixel falls inside the reference crop window, and take the bounding box of their nadir10 pixels (no RPCs needed — the match points *are* the mapping between the two image spaces):

In [ ]:
import os

import numpy as np

XOFF, YOFF, W, H = 5879, 13107, 12981, 11894  # reference (nadir13) crop win

ba_match_fn = os.path.expanduser(
    f"{directory}ba/run-1030010002B7D800_P002_corr__1030010003CAF100_P002_corr-clean.match"
)

# get_match_point_df() parses ASP's binary match format into x1/y1 (nadir13)
# and x2/y2 (nadir10) pixel coordinates
match_df = stereo_plotter.get_match_point_df(match_point_fn=ba_match_fn)

inwin = match_df[
    match_df.x1.between(XOFF, XOFF + W) & match_df.y1.between(YOFF, YOFF + H)
]

pad = 100
x0, y0 = int(inwin.x2.min()) - pad, int(inwin.y2.min()) - pad
x1, y1 = int(inwin.x2.max()) + pad, int(inwin.y2.max()) + pad
print(f"{len(inwin)} of {len(match_df)} clean match points fall in the reference crop win")
print(f"nadir10 crop win: --left-image-crop-win {x0} {y0} {x1 - x0} {y1 - y0}")

Run the three pairs and mosaic the DEMs:

```bash
$ cd atlanta_mvs

$ N13=1030010002B7D800_P002; N10=1030010003CAF100_P002; N16=1030010002649200_P001

# nadir13-nadir10 and nadir13-nadir16: reference crop win
$ for right in $N10 $N16; do
    out=stereo_pair_13_$([ $right = $N10 ] && echo 10 || echo 16)
    parallel_stereo \
      --stereo-algorithm asp_mgm --subpixel-mode 9 \
      --alignment-method affineepipolar \
      --left-image-crop-win 5879 13107 12981 11894 \
      --bundle-adjust-prefix ba/run \
      ${N13}_corr.tif ${right}_corr.tif ${N13}.xml ${right}.xml ${out}/run
    point2dem --tr 1.9 --t_srs EPSG:32616 --errorimage ${out}/run-PC.tif
  done

# nadir10-nadir16: crop win derived from the match points above
$ parallel_stereo \
    --stereo-algorithm asp_mgm --subpixel-mode 9 \
    --alignment-method affineepipolar \
    --left-image-crop-win 5540 13223 13206 12931 \
    --bundle-adjust-prefix ba/run \
    ${N10}_corr.tif ${N16}_corr.tif ${N10}.xml ${N16}.xml stereo_pair_10_16/run
$ point2dem --tr 1.9 --t_srs EPSG:32616 --errorimage stereo_pair_10_16/run-PC.tif

# Blend the three pairwise DEMs (dem_mosaic averages where they overlap)
$ dem_mosaic \
    stereo_pair_13_10/run-DEM.tif stereo_pair_13_16/run-DEM.tif \
    stereo_pair_10_16/run-DEM.tif -o pairwise_mosaic
$ mv pairwise_mosaic-tile-0.tif pairwise_mosaic-DEM.tif
```

---

## Comparison: MVS vs. Pairwise + Mosaic

Both DEMs were produced from the same five-scene bundle adjustment, the same three images, the same matched ground crops, and the same correlation settings — the only difference is *joint triangulation* versus *independent pairs blended after gridding*.

### Coverage and DEM-to-DEM difference

In [ ]:
import matplotlib.pyplot as plt

from asp_plot.utils import Raster, nmad

mvs_dem_fn = os.path.expanduser(f"{directory}stereo_mvs3/run-DEM.tif")
pairwise_dem_fn = os.path.expanduser(f"{directory}pairwise_mosaic-DEM.tif")

mvs_dem = Raster(mvs_dem_fn)
pairwise_dem = Raster(pairwise_dem_fn)

gsd = mvs_dem.get_gsd()
for name, r in [("MVS", mvs_dem), ("Pairwise+mosaic", pairwise_dem)]:
    data = r.read_array()
    valid_km2 = data.count() * gsd**2 / 1e6
    pct = 100 * data.count() / data.size
    print(f"{name:16s} valid area {valid_km2:6.2f} km2 ({pct:.1f}% of its grid)")

# MVS minus pairwise mosaic, on the MVS grid
diff = pairwise_dem.compute_difference(mvs_dem_fn)

med, spread = np.ma.median(diff), nmad(diff.compressed())
print(f"\nMVS - pairwise: median {med:+.2f} m, NMAD {spread:.2f} m")

fig, axes = plt.subplots(1, 2, figsize=(12, 5), width_ratios=[1.2, 1])
im = axes[0].imshow(diff, cmap="RdBu", vmin=-2, vmax=2)
fig.colorbar(im, ax=axes[0], label="MVS - pairwise mosaic (m)", shrink=0.8)
axes[0].set_title("DEM difference")
axes[0].set_axis_off()
axes[1].hist(diff.compressed(), bins=256, range=(-5, 5), color="0.4")
axes[1].axvline(0, color="k", lw=0.5)
axes[1].set_xlabel("MVS - pairwise mosaic (m)")
axes[1].set_title(f"median {med:+.2f} m, NMAD {spread:.2f} m")
fig.tight_layout()

### ICESat-2 residuals

Validate both DEMs against the **same ICESat-2 ATL06-SR sample** (the parquet cache written when the report below was first generated — loading it replays the exact points, bypassing the SlideRule request). Water returns are filtered with ESA WorldCover before differencing:

In [ ]:
from asp_plot.altimetry import Altimetry

parquet = {"all": os.path.expanduser(f"{directory}atl06sr_all.parquet")}

results = {}
for name, dem_fn in [("MVS", mvs_dem_fn), ("Pairwise+mosaic", pairwise_dem_fn)]:
    icesat = Altimetry(directory=directory, dem_fn=dem_fn)
    icesat.load_atl06sr_from_parquet(parquet)
    icesat.filter_esa_worldcover(filter_out="water")
    icesat.atl06sr_to_dem_dh()
    results[name] = icesat

import pandas as pd

summary = pd.DataFrame(
    {
        name: {
            "n points": len(dh := icesat.atl06sr_processing_levels_filtered["all"]["icesat_minus_dem"].dropna()),
            "median (m)": round(dh.median(), 3),
            "NMAD (m)": round(nmad(dh), 3),
            "p16 (m)": round(dh.quantile(0.16), 3),
            "p84 (m)": round(dh.quantile(0.84), 3),
        }
        for name, icesat in results.items()
    }
)
summary

In [ ]:
# Per-landcover-class histograms for each DEM
for name, icesat in results.items():
    print(f"--- {name} ---")
    icesat.histogram_by_landcover(key="all")

### Takeaways

- **Coverage**: the pairwise mosaic fills slightly more ground — where correlation fails for one pair, another pair can still contribute — while the MVS DEM leaves holes wherever the joint solution lacks support.
- **Agreement**: over most of the scene the two DEMs agree tightly; the differences concentrate on buildings, tree canopy, and the areas only covered by the weak 5.3° pair.
- **ICESat-2**: both DEMs sit close to the altimetry; compare the median bias and NMAD in the summary table above — with a same-pass trio and shared cameras, joint triangulation and pairwise blending land within centimeters to a few decimeters of each other.

Which strategy wins in general — and whether more scenes help — is the subject of the deferred benchmarking issue [#169](https://github.com/uw-cryo/asp_plot/issues/169), which will score scene combinations and processing flows against ICESat-2 after `pc_align`.

---

## Full Report Generation

Generate the standard `asp_plot` PDF report for the multi-view run. The Input Scenes, Match Points, and Disparity sections contain one figure per `run-pair*/` pair, the Stereo Geometry section is scoped to the run's three cameras, and the ICESat-2 altimetry comparison (including `pc_align`) runs automatically for Earth data.

#### See the resulting [report](https://asp-plot.readthedocs.io/en/latest/_static/reports/WorldView_Atlanta_MVS-asp-plot-report.pdf).

In [ ]:
!asp_plot \
    --directory $directory \
    --bundle_adjust_directory ba/ \
    --stereo_directory stereo_mvs3/ \
    --report_filename ../../reports/WorldView_Atlanta_MVS-asp-plot-report.pdf

---

## Next Steps

- **Scene selection**: see [`worldview_spacenet_atlanta_stereo_scene_selection.ipynb`](https://asp-plot.readthedocs.io/en/latest/examples/notebooks/worldview_spacenet_atlanta_stereo_scene_selection.html) for the pair-ranking analysis behind these scenes.
- **Benchmarking**: [issue #169](https://github.com/uw-cryo/asp_plot/issues/169) tracks the systematic MVS vs. n-pair comparison across scene combinations, scored against ICESat-2 after `pc_align`.
- **Another multi-view example**: the [Pléiades Neo Marseille tri-stereo notebook](https://asp-plot.readthedocs.io/en/latest/examples/notebooks/pleiades_neo_marseille_tristereo.html) runs the same joint-MVS workflow on an Airbus same-pass tri-stereo collect.
- **Jitter correction**: for highest-quality DEMs, apply jitter correction following [ASP Documentation](https://stereopipeline.readthedocs.io/en/latest/tools/jitter_solve.html).